In [ ]:
# YOLOv8 Object Detection - Betel Insects Dataset
# Classes: Green-Leafhopper, aphid, beetle, grasshopper
# Features: Training, Evaluation, OOD Detection (Mahalanobis Distance)
# Platform: Kaggle Notebook
# =============================================================================

# ─────────────────────────────────────────────
# CELL 1: Install Dependencies
# ─────────────────────────────────────────────
!pip install ultralytics onnx onnxruntime onnxscript -q
import os, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
import yaml
import cv2
import torch
from collections import defaultdict

from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.covariance import EmpiricalCovariance

warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Configuration
# ─────────────────────────────────────────────

# Dataset & output paths
DATASET_ROOT   = Path("/kaggle/input/datasets/amandapriyashantha/betel-pets-obj-01")
WORKING_DIR    = Path("/kaggle/working")
FILTERED_DIR   = WORKING_DIR / "betel_filtered"
OUTPUT_DIR     = WORKING_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Classes to KEEP (indices in the ORIGINAL dataset must be discovered first)
KEEP_CLASSES   = ["Green-Leafhopper", "aphid", "beetle", "grasshopper"]
REMOVE_CLASSES = ["armyworm", "sawfly"]

# Training hyper-parameters
IMG_SIZE  = 640
EPOCHS    = 25
BATCH     = 16
DEVICE    = 0 if torch.cuda.is_available() else "cpu"
MODEL_CFG = "yolov8m.pt"           # medium backbone – good balance for Kaggle P100

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device         : {DEVICE}")
print(f"Dataset root   : {DATASET_ROOT}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Discover Original Class Mapping
# ─────────────────────────────────────────────

orig_yaml_path = DATASET_ROOT / "data.yaml"
with open(orig_yaml_path) as f:
    orig_cfg = yaml.safe_load(f)

orig_classes = orig_cfg["names"]           # list of class names
print("Original classes:", orig_classes)

# Map: original index → class name
orig_id2name = {i: n for i, n in enumerate(orig_classes)}
orig_name2id = {n: i for i, n in enumerate(orig_classes)}

# New consecutive mapping (keep order)
new_classes  = [c for c in orig_classes if c in KEEP_CLASSES]
new_name2id  = {n: i for i, n in enumerate(new_classes)}
new_id2name  = {i: n for i, n in enumerate(new_classes)}

# Set of original IDs to KEEP
keep_orig_ids = {orig_name2id[c] for c in KEEP_CLASSES if c in orig_name2id}
# Remap: original_id → new_id
remap = {oid: new_name2id[orig_id2name[oid]] for oid in keep_orig_ids}

print("New classes    :", new_classes)
print("Remap table    :", remap)